In [1]:
import os
import mne
import numpy as np
import pandas as pd
import pickle
from typing import Dict, List, Any, Tuple
from tqdm import tqdm
from scipy import signal
import traceback
import torch
import torch.nn.functional as F


# Configuration constants
SESSION_CONFIGS = {
    'audio': {
        'class_ranges': {
            'bad': '1', 'go': '2', 'good': '3', 'happy': '4', 'hello': '5',
            'help': '6', 'no': '7', 'stop': '8', 'thanks': '9', 'yes': '10'
        },
        'baseline_mode': 'eyes_closed',
        'baseline_codes': ['102']  # Changed to string
    },
    'read': {
        'class_ranges': {
            'bad': (1, 10), 'go': (11, 20), 'good': (21, 30), 'happy': (31, 40),
            'hello': (41, 50), 'help': (51, 60), 'no': (61, 70), 'stop': (71, 80),
            'thanks': (81, 90), 'yes': (91, 100)
        },
        'baseline_mode': 'eyes_open',
        'baseline_codes': ['101']
    }
}


def compute_subject_baseline(raw, annotations, baseline_codes: List[str]) -> np.ndarray:
    """Compute baseline using specified annotation codes"""
    baseline_indices = []
    
    for i, ann in enumerate(annotations):
        if ann['description'] in baseline_codes:
            start_sample = int(ann['onset'] * raw.info['sfreq'])
            duration_samples = int(ann['duration'] * raw.info['sfreq'])
            baseline_indices.extend(range(start_sample, start_sample + duration_samples))
    
    if not baseline_indices:
        print(f"Warning: No baseline events found for codes {baseline_codes}")
        return None
    
    baseline_data, _ = raw[:, baseline_indices]
    return np.mean(baseline_data, axis=1)


def build_event_mapping(annotations: List, class_ranges: Dict, 
                       session_type: str) -> Tuple[Dict, Dict, Dict]:
    """Build mappings between annotations, classes, and standardized event IDs"""
    standardized_event_id = {}
    file_event_id = {}
    class_mapping = {}  # Maps annotation description to (class_name, standardized_id)
    next_event_id = 1000
    
    print(f"\nFound {len(annotations)} annotations in file")
    if annotations:
        unique_descriptions = set([ann['description'] for ann in annotations])
        print(f"Unique annotation descriptions: {sorted(unique_descriptions)}")
    
    if session_type == 'audio':
        for class_name, event_str in class_ranges.items():
            for ann in annotations:
                if ann['description'] == event_str:
                    key = f"{class_name}_{event_str}"
                    if key not in standardized_event_id:
                        standardized_event_id[key] = next_event_id
                        file_event_id[key] = next_event_id
                        next_event_id += 1
                    class_mapping[event_str] = (class_name, standardized_event_id[key])
                    break
    else:  # read session
        for class_name, (start_code, end_code) in class_ranges.items():
            for code in range(start_code, end_code + 1):
                code_str = str(code)
                for ann in annotations:
                    if ann['description'] == code_str:
                        key = f"{class_name}_{code_str}"
                        if key not in standardized_event_id:
                            standardized_event_id[key] = next_event_id
                            file_event_id[key] = next_event_id
                            next_event_id += 1
                        class_mapping[code_str] = (class_name, standardized_event_id[key])
                        break
    
    print(f"\nTotal class mappings created: {len(class_mapping)}")
    print(f"Classes found: {[class_mapping[code][0] for code in class_mapping.keys()]}")
    
    return standardized_event_id, file_event_id, class_mapping


def extract_frequency_bands(eeg_data: np.ndarray, sfreq: float = 250) -> Dict[str, np.ndarray]:
    """Extract frequency band features"""
    bands = {
        't1': (4, 6),
        't2': (6, 8),
        'a1': (8, 10),
        'a2': (10, 13),
        'b1': (13, 20),
        'b2': (20, 30),
        'g1': (30, 40),
        'g2': (40, 50)
    }
    band_features = {}
    
    for band_name, (low_freq, high_freq) in bands.items():
        if high_freq > sfreq / 2:
            continue
        b, a = signal.butter(4, [low_freq, high_freq], btype='band', fs=sfreq)
        filtered_data = np.zeros_like(eeg_data)
        for ch in range(eeg_data.shape[0]):
            filtered_data[ch] = signal.filtfilt(b, a, eeg_data[ch])
        band_features[f'mean_{band_name}'] = np.mean(filtered_data, axis=1)
        band_features[f'std_{band_name}'] = np.std(filtered_data, axis=1)
        band_features[f'power_{band_name}'] = np.mean(filtered_data ** 2, axis=1)
    
    return band_features


def create_word_level_structure(eeg_data: np.ndarray, 
                               word_label: str,
                               frequency_bands: Dict[str, np.ndarray],
                               channel_names: List[str],
                               sfreq: float = 250,
                               metadata: Dict = None) -> Dict:
    """Create word-level EEG structure"""
    word_obj = {
        'content': word_label,
        'word_level_EEG': {
            'raw_eeg': eeg_data,
            'frequency_bands': frequency_bands,
            'GD': frequency_bands,
            'FFD': frequency_bands,
            'TRT': frequency_bands
        },
        'nFixations': 1,
        'channels': channel_names,
        'channel_indices': list(range(len(channel_names))),
        'sampling_rate': sfreq,
        'duration_ms': 1000,
        'n_channels': eeg_data.shape[0],
        'n_timepoints': eeg_data.shape[1]
    }
    if metadata:
        word_obj.update(metadata)
    return word_obj


def create_events_from_annotations(annotations, class_mapping, sfreq):
    """Create events array from annotations using class mapping"""
    events = []
    for ann in annotations:
        desc = ann['description']
        if desc in class_mapping:
            class_name, event_id = class_mapping[desc]
            onset_sample = int(ann['onset'] * sfreq)
            events.append([onset_sample, 0, event_id])
    return np.array(events) if events else np.array([])


def process_edf_to_pickle(edf_paths: List[str], 
                          output_dir: str,
                          session_type: str = 'read',
                          task_name: str = 'single_word_eeg') -> Dict:
    
    if session_type not in SESSION_CONFIGS:
        raise ValueError(f"session_type must be 'audio' or 'read', got '{session_type}'")
    
    config = SESSION_CONFIGS[session_type]
    class_ranges = config['class_ranges']
    baseline_codes = config['baseline_codes']
    
    dataset_dict = {}
    
    print(f"Processing {len(edf_paths)} EDF files...")
    
    for edf_path in tqdm(edf_paths, desc="Processing EDF files"):
        subject_id = os.path.basename(edf_path).split('_')[0]
        if subject_id not in dataset_dict:
            dataset_dict[subject_id] = []
        
        try:
            raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
            
            # ---- Remove Trigger channel if exists ----
            if 'Trigger' in raw.ch_names:
                raw.drop_channels(['Trigger'])
                print(f"Removed 'Trigger' channel. Remaining channels: {len(raw.ch_names)}")
            
            orig_sfreq = raw.info['sfreq']
            print(f"Loaded {len(raw.ch_names)} channels at {orig_sfreq} Hz")
            
            annotations = raw.annotations
            
            _, file_event_id, class_mapping = build_event_mapping(annotations, class_ranges, session_type)
            if not class_mapping:
                print(f"Warning: No class mappings created for {edf_path}")
                continue
            
            events = create_events_from_annotations(annotations, class_mapping, orig_sfreq)
            if len(events) == 0:
                print(f"Warning: No valid events created for {edf_path}, skipping...")
                continue
            
            # Filtering
            raw.filter(l_freq=1., h_freq=50., fir_design='firwin', verbose=False)
            
            # Resample
            raw, events = raw.resample(250, events=events)
            
            # Baseline
            baseline_mean = compute_subject_baseline(raw, annotations, baseline_codes)
            
            # Create epochs
            epochs = mne.Epochs(
                raw, events=events, event_id=file_event_id,
                tmin=0, tmax=0.999, baseline=None, preload=True, verbose=False
            )
            if baseline_mean is not None:
                epochs._data -= baseline_mean[:, np.newaxis]
            
            # Process each epoch
            for epoch_idx in range(len(epochs)):
                eeg_data = epochs[epoch_idx].get_data()[0]
                event_id = epochs.events[epoch_idx, 2]
                
                class_name = None
                for code, (cn, std_id) in class_mapping.items():
                    if std_id == event_id:
                        class_name = cn
                        break
                if class_name is None:
                    continue
                
                frequency_bands = extract_frequency_bands(eeg_data, sfreq=250)
                metadata = {
                    'subject_id': subject_id,
                    'file_path': edf_path,
                    'event_id': event_id,
                    'class_label': class_name,
                    'trial_index': epoch_idx,
                    'session_type': session_type,
                    'timestamp': epochs.times.tolist(),
                    'original_shape': eeg_data.shape,
                    'sampling_rate': 250
                }
                
                word_obj = create_word_level_structure(
                    eeg_data=eeg_data,
                    word_label=class_name,
                    frequency_bands=frequency_bands,
                    channel_names=raw.ch_names,
                    sfreq=250,
                    metadata=metadata
                )
                
                dataset_dict[subject_id].append(word_obj)
            
            print(f"✓ Added {len(epochs)} word instances for subject {subject_id}")
        
        except Exception as e:
            print(f"✗ Error processing {edf_path}: {str(e)}")
            traceback.print_exc()
            continue
    
    # Save pickle
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f'{task_name}.pickle')
    with open(output_path, 'wb') as handle:
        pickle.dump(dataset_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
    print(f"\nSaved dataset to: {output_path}")
    return dataset_dict


def create_simplified_pickle_format(dataset_dict: Dict, 
                                   output_path: str,
                                   include_raw_eeg: bool = True,
                                   include_frequency_features: bool = True) -> None:
    """Simplified format"""
    
    simplified_data = {'X_raw': [], 'X_features': [], 'y': [], 'subjects': [], 'word_labels': [], 'channels': [], 'metadata': []}
    
    all_classes = set()
    for subject_id, instances in dataset_dict.items():
        for instance in instances:
            all_classes.add(instance['content'])
    if not all_classes:
        print("Warning: No data found in dataset_dict!")
        return
    
    class_to_idx = {cls: idx for idx, cls in enumerate(sorted(all_classes))}
    
    for subject_id, word_instances in dataset_dict.items():
        for instance in word_instances:
            if include_raw_eeg:
                simplified_data['X_raw'].append(instance['word_level_EEG']['raw_eeg'])
            
            features = []
            if include_frequency_features:
                freq_bands = instance['word_level_EEG']['frequency_bands']
                band_features = []
                for band in ['t1','t2','a1','a2','b1','b2','g1','g2']:
                    if f'mean_{band}' in freq_bands:
                        band_features.append(freq_bands[f'mean_{band}'])
                if band_features:
                    features.append(np.concatenate(band_features))
            if features:
                simplified_data['X_features'].append(np.concatenate(features))
                simplified_data['y'].append(class_to_idx[instance['content']])
                simplified_data['subjects'].append(subject_id)
                simplified_data['word_labels'].append(instance['content'])
                simplified_data['channels'].append(instance['channels'])
                simplified_data['metadata'].append({
                    'subject_id': instance.get('subject_id',''),
                    'n_channels': instance.get('n_channels',0),
                    'sampling_rate': instance.get('sampling_rate',0),
                    'duration_ms': instance.get('duration_ms',0),
                    'session_type': instance.get('session_type','')
                })
    
    # Convert to arrays
    if simplified_data['X_raw']:
        simplified_data['X_raw'] = np.array(simplified_data['X_raw'])
    if simplified_data['X_features']:
        simplified_data['X_features'] = np.array(simplified_data['X_features'])
    if simplified_data['y']:
        simplified_data['y'] = np.array(simplified_data['y'])
    
    # Save
    with open(output_path, 'wb') as f:
        pickle.dump(simplified_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"\nSaved simplified format to: {output_path}")


# ---- Example main usage ----
if __name__ == "__main__":
    # select "audio" or "read" below
    subjects = ['s2','s3','s4','s5','s8','s10','s12','s7','s9','s11','s6']
    edf_files = [f"{sub}_audio{i} Data.edf" for sub in subjects for i in [1,2]]
    existing_files = [f for f in edf_files if os.path.exists(f)]
    
    if not existing_files:
        print("No EDF files found!")
        exit(1)
    
    dataset_dict = process_edf_to_pickle(
        edf_paths=existing_files,
        output_dir="./processed_data/pickle/",
        session_type='audio',
        task_name='single_word_classification'
    )

Processing 22 EDF files...


Processing EDF files:   0%|          | 0/22 [00:00<?, ?it/s]

Removed 'Trigger' channel. Remaining channels: 64
Loaded 64 channels at 1000.0 Hz

Found 208 annotations in file
Unique annotation descriptions: ['1', '10', '101', '102', '103', '104', '105', '2', '3', '4', '5', '6', '7', '8', '9']

Total class mappings created: 10
Classes found: ['bad', 'go', 'good', 'happy', 'hello', 'help', 'no', 'stop', 'thanks', 'yes']


Processing EDF files:   5%|▍         | 1/22 [00:12<04:15, 12.17s/it]

✓ Added 200 word instances for subject s2
Removed 'Trigger' channel. Remaining channels: 64
Loaded 64 channels at 1000.0 Hz

Found 208 annotations in file
Unique annotation descriptions: ['1', '10', '101', '102', '103', '104', '105', '2', '3', '4', '5', '6', '7', '8', '9']

Total class mappings created: 10
Classes found: ['bad', 'go', 'good', 'happy', 'hello', 'help', 'no', 'stop', 'thanks', 'yes']


...

Processing EDF files:  95%|█████████▌| 21/22 [04:26<00:12, 12.38s/it]

✓ Added 200 word instances for subject s6
Removed 'Trigger' channel. Remaining channels: 64
Loaded 64 channels at 1000.0 Hz

Found 209 annotations in file
Unique annotation descriptions: ['1', '10', '100015', '101', '102', '103', '104', '105', '2', '3', '4', '5', '6', '7', '8', '9']

Total class mappings created: 10
Classes found: ['bad', 'go', 'good', 'happy', 'hello', 'help', 'no', 'stop', 'thanks', 'yes']


Processing EDF files: 100%|██████████| 22/22 [04:42<00:00, 12.85s/it]

✓ Added 200 word instances for subject s6



Saved dataset to: ./processed_data/pickle/single_word_classification.pickle

Saved simplified format to: ./processed_data/simplified_dataset.pickle
